In [ ]:
# ── SETUP ────────────────────────────────────────────────
# Before running for the first time, create the token file:
# 1. Generate a token: https://dar.elter-ri.eu/account/settings/applications/tokens/new/
# 2. Run in the terminal:
#    echo "your-token-here" > .elter-dar-access-token
# Put .elter-dar-access-token in .gitignore!!!
# Possible complications with the jq package
# For me, it worked with:
#   curl -L https://github.com/stedolan/jq/releases/download/jq-1.6/jq-linux64 -o ~/jq
#   chmod +x ~/jq
#   mv ~/jq /usr/local/bin/jq
# and then, if needed, in every cell:
#   export PATH="$PATH:$HOME" (or wherever jq is located, here in home)
# For BASE_DIR, and in every cell with "cd ...", personalize the path!
#
# NOTE: Sorting of IDs into site folders does not work 100% reliably yet,
# since site names are sometimes written inconsistently. Therefore,
# please double-check whether ALL dataset IDs belonging to a site have
# actually been processed/found — some may have ended up under a
# differently spelled site folder.
# ───────────────────────────────────────────────────────────

In [1]:
from __future__ import annotations
from pathlib import Path
import subprocess
import zipfile
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

BASE_DIR     = Path("/home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools")
TOKEN_FILE   = BASE_DIR / ".elter-dar-access-token"
IDS_FILE     = BASE_DIR / "eLTER-Data-Call-2025-Uploads.md"
METADATA_CSV = BASE_DIR / "metadata.csv"
DOWNLOAD_DIR = BASE_DIR / "downloads_IDs"
DOWNLOAD_DIR.mkdir(exist_ok=True)

print(f"BASE_DIR:  {BASE_DIR}")
print(f"Token:     {TOKEN_FILE.name}")

BASE_DIR:  /home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools
Token:     .elter-dar-access-token


In [3]:
%%bash
cd /home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools

IDS_FILE="eLTER-Data-Call-2025-Uploads.md"

# Skip if the file was already created today
if [ -f "$IDS_FILE" ] && [ "$(date -r $IDS_FILE +%Y-%m-%d)" = "$(date +%Y-%m-%d)" ]; then
    echo "IDs already fetched today — skipping!"
    wc -l "$IDS_FILE"
    exit 0
fi

ACCESS_TOKEN=$(grep -Ev "^\s*(#|$)" .elter-dar-access-token | head -n1)

echo "Fetching IDs from DAR..."

# Fetch all submitted IDs, paginated
for page in $(seq 1 300); do
    result=$(curl -fsSL \
        "https://dar.elter-ri.eu/api/requests/?page=${page}&size=10&status=submitted" \
        -H "Authorization: Bearer ${ACCESS_TOKEN}" | \
        jq -r '.hits.hits[].topic.datasets_draft // empty')
    
    if [ -z "$result" ]; then
        break
    fi
    echo "$result"
done > eLTER-Data-Call-2025-Uploads.md

echo "Done!"
wc -l eLTER-Data-Call-2025-Uploads.md

IDs fetchen vom DAR...
Fertig!
224 eLTER-Data-Call-2025-Uploads.md


In [11]:
%%bash
cd /home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools
bash elter-dar-query.sh > metadata.csv
echo "Done! Metadata saved in metadata.csv"
wc -l metadata.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12756  100 12756    0     0  11949      0  0:00:01  0:00:01 --:--:-- 11955
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11062  100 11062    0     0  12746      0 --:--:-- --:--:-- --:--:-- 12744
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7276  100  7276    0     0   7317      0 --:--:-- --:--:-- --:--:--  7312
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7615  100  7615    0     0   8068      0 --:--:-- --:--:-- --:--:--  8066
  % Total    % Received % Xferd  Average Speed   Tim

Fertig! Metadaten gespeichert in metadata.csv
450 metadata.csv


In [12]:
df_meta = pd.read_csv(METADATA_CSV)
print(f"Total datasets: {len(df_meta)}")
display(df_meta.head(10))

# Optional: filter by SO type
FILTER_BY_SO = []  # e.g. ["SOHYD_004", "SOATM_027"] or [] for all
if FILTER_BY_SO:
    df_filtered = df_meta[df_meta[".metadata.datasetType.datasetTypeCode"].isin(FILTER_BY_SO)]
    print(f"After filter: {len(df_filtered)} datasets")
    display(df_filtered)

Datensätze gesamt: 224


,.id,.metadata.titles[0].titleText,.metadata.siteReferences[0].siteName,.metadata.siteReferences[0].siteID,.metadata.datasetType.datasetTypeCode,.metadata.descriptions[0].descriptionText,.metadata.responsibleOrganizations[0].organizationName,.access.embargo.active,.access.embargo.reason
0,xpwd0-rxm33,Monitoring of Soil Moisture and temperature_RE...,Companhia das Lezírias - Portugal,ef87d551-9bae-467f-9c5d-b538b8206b0b,SOHYD_168,This is part of the RENEWAL project (following...,University of Lisbon_cE3c - Centre for Ecology...,False,NaN
1,95ydm-51y27,Monitoring Solar Radiation_Renewal Project -> ...,Companhia das Lezírias - Portugal,ef87d551-9bae-467f-9c5d-b538b8206b0b,SOATM_028,Photosynthetically Active Radiation (PAR) Smar...,University of Lisbon_cE3c - Centre for Ecology...,False,NaN
2,69v18-n1p06,Monitoring Solar Radiation -> Companhia das Le...,Companhia das Lezírias - Portugal,ef87d551-9bae-467f-9c5d-b538b8206b0b,SOATM_028,Solar radiation measured with a weather statio...,University of Lisbon_cE3c - Centre for Ecology...,False,NaN
3,zk6wk-edw78,Monitoring Solar Radiation -> Herdade da Ribei...,Herdade da Ribeira Abaixo - Portugal,c887da3c-776c-4b42-a5be-7dd69c26f27b,SOATM_028,Solar radiation measured with a weather statio...,University of Lisbon_cE3c - Centre for Ecology...,False,NaN
4,npdt4-w6h66,Metereological data-> Herdade da Ribeira Abaix...,Herdade da Ribeira Abaixo - Portugal,c887da3c-776c-4b42-a5be-7dd69c26f27b,SOATM_027,"Metereological data, e.g. air temperature, air...",University of Lisbon_cE3c - Centre for Ecology...,False,NaN
5,1gae8-1rc68,Metereological data-> Companhia das Lezírias ...,Companhia das Lezírias - Portugal,ef87d551-9bae-467f-9c5d-b538b8206b0b,SOATM_027,"Metereological data, e.g., air temperature, ai...",University of Lisbon_cE3c - Centre for Ecology...,False,NaN
6,djqd2-27768,HU_Kiskun-LTER_Fulophaza_SOATM_027_2001_2024,KISKUN LTER - Hungary,124f227a-787d-4378-bc29-aa94f29e1732,SOATM_027,"Daily meteorological data, including air tempe...",HUN-REN Centre for Ecological Research,False,NaN
7,j0b6g-5af87,HU_Kiskun-LTER_Restoration_Seeding_SOBIO_017_2...,KISKUN LTER - Hungary,124f227a-787d-4378-bc29-aa94f29e1732,SOBIO_017,"In Hungary, common milkweed (Asclepias syriaca...",HUN-REN Centre for Ecological Research,False,NaN
8,5sr7b-zee43,HU_Kiskun-LTER_Restoration_Asclepias_SOBIO_017...,KISKUN LTER - Hungary,124f227a-787d-4378-bc29-aa94f29e1732,SOBIO_017,"In Hungary, common milkweed (Asclepias syriaca...",HUN-REN Centre for Ecological Research,False,NaN
9,eavgt-wqa88,HU_Kiskun-LTER_Kiskun Restoration Experiments_...,"Kiskun Restoration Experiments, KISKUN LTER - ...",c7a1d72c-7296-49e7-813a-890a11cf0ae9,SOBIO_017,Yearly species cover data (%) from 2012 to 201...,HUN-REN Centre for Ecological Research,False,NaN


In [ ]:
%%bash
cd /home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools
# Choose here:
# Option 1: All IDs (default, skips already-downloaded datasets)
bash elter-dar-download.sh

# Option 2: Single IDs from the SINGLE_IDS list in the .sh file
# bash elter-dar-download.sh --single

# Option 3: One specific ID directly
# bash elter-dar-download.sh xpwd0-rxm33

# To re-download and overwrite already existing data, add: --existing=overwrite
# e.g.: bash elter-dar-download.sh --existing=overwrite